# **Limpeza e Extração de Dados das Súmulas CBF**

## **1. Configuração do Ambiente**

In [10]:
import re
import unicodedata
from pathlib import Path

import pandas as pd
import pdfplumber

## **2. Configuração**

Carrega o índice de súmulas gerado no notebook 0. Cada linha do índice corresponde a uma súmula baixada, com metadados de jogo e caminho do PDF.

Este notebook produz duas bases limpas:

- `sumulas_clean.csv`: formato largo, com duas linhas por jogo, preservado para compatibilidade;
- `card_reasons_long.csv`: formato longo, com uma linha por cartão. Esta é a entrada principal do notebook 2.

In [11]:
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DATA_PATH = PROJECT_ROOT / "data"
RAW_PATH = DATA_PATH / "raw" / "match-reports"
PROCESSED_PATH = DATA_PATH / "processed"
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

INDEX_PATH = RAW_PATH / "match_reports_index.csv"
OUTPUT_PATH = PROCESSED_PATH / "sumulas_clean.csv"
CARD_REASONS_LONG_PATH = PROCESSED_PATH / "card_reasons_long.csv"
SUCCESS_DOWNLOAD_STATUSES = {"exists", "downloaded"}

index = pd.read_csv(INDEX_PATH)
index = index[index["download_status"].isin(SUCCESS_DOWNLOAD_STATUSES)].reset_index(drop=True)
index.head(2)

,year,competition_id,round,game_id,game_number,home_team,away_team,match_date,match_time,venue,document_title,document_title_normalized,document_url,file_path,download_status
0,2020,12464,1,815072,1,Flamengo,Atlético Mineiro,09/08/2020,16:00,Maracanã - Rio de Janeiro - RJ,Súmula,sumula,https://conteudo.cbf.com.br/sumulas/2020/1421s...,../data/raw/match-reports/2020/round-01/2020_r...,downloaded
1,2020,12464,1,815073,2,Botafogo de Futebol E Regatas,Esporte Clube Bahia,30/09/2020,21:30,Nilton Santos - Rio de Janeiro - RJ,Súmula,sumula,https://conteudo.cbf.com.br/sumulas/2020/1422s...,../data/raw/match-reports/2020/round-01/2020_r...,downloaded


## **3. Extração de Texto dos PDFs**

Cada súmula é um PDF de 2–3 páginas. O texto é extraído com `pdfplumber` e as páginas são concatenadas em uma única string para simplificar o parsing com regex.

In [12]:
def resolve_pdf_path(file_path: str) -> Path:
    path = Path(file_path)
    if path.is_absolute():
        return path
    candidate = (PROJECT_ROOT / path).resolve()
    if candidate.exists():
        return candidate
    return (PROJECT_ROOT / "notebooks" / path).resolve()


def extract_pdf_text(file_path: str) -> str:
    with pdfplumber.open(resolve_pdf_path(file_path)) as pdf:
        return "\n".join(page.extract_text() or "" for page in pdf.pages)

## **4. Funções de Parsing**

### 4.1 Utilitários de Normalização

O matching de nomes de times usa normalização (sem acento, minúsculo) para lidar com variações entre o índice e o texto do PDF — por exemplo, `"Fortaleza Esporte Clube"` no índice vs. `"Fortaleza/CE"` no PDF.

In [13]:
def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFKD", str(text))
    text = "".join(c for c in text if not unicodedata.combining(c))
    return text.casefold().strip()


def _prep_for_match(text: str) -> str:
    n = normalize_text(text)
    n = re.sub(r"/[a-z]{2,3}\s*$", "", n).rstrip()
    return re.sub(r"[^\w\s]", "", n).strip()


def identify_team(line: str, home_team: str, away_team: str) -> str | None:
    line_clean = _prep_for_match(line)
    home_n = re.sub(r"[^\w\s]", "", normalize_text(home_team)).strip()
    away_n = re.sub(r"[^\w\s]", "", normalize_text(away_team)).strip()

    def score(team_n: str, line_c: str) -> float:
        if line_c.endswith(team_n):
            return 3.0
        tokens = [t for t in team_n.split() if len(t) > 3]
        last3 = " ".join(line_c.split()[-3:])
        for tok in tokens:
            if line_c.endswith(tok):
                return 1.0
        for tok in tokens:
            if tok in last3:
                return 0.5
        return 0.0

    hs = score(home_n, line_clean)
    as_ = score(away_n, line_clean)
    if hs > as_:
        return home_team
    if as_ > hs:
        return away_team
    return None

### 4.2 Campos Simples

Extrai árbitro e data diretamente do texto do PDF via regex.

In [14]:
def parse_arbitro(text: str) -> str | None:
    m = re.search(r"Arbitro:\s*([^(\n]+)", text)
    return m.group(1).strip() if m else None


def parse_match_date(match_date_raw: str) -> str:
    return pd.to_datetime(str(match_date_raw).strip(), format="%d/%m/%Y").strftime("%Y-%m-%d")

### 4.3 Parsing de Cartões

Cada entrada de cartão no PDF tem um cabeçalho com timestamp e time, seguido de um campo `Motivo:` potencialmente multi-linha.

A limpeza agora gera dois níveis:

- `reason`: texto completo do motivo, já sem o código inicial (`A1.4.`, `V1.`, etc.);
- `reason_category` e `reason_detail`: separação aproximada entre categoria padronizada da CBF e descrição livre do árbitro.

Essa tabela longa é a base correta para NLP no notebook 2.

In [15]:
TIMESTAMP_RE = re.compile(r"^(?P<minute>\d{2}):(?P<second>\d{2})\s+(?P<half>[12]T)\s+")
SKIP_RE = re.compile(r"^(Tempo 1T|NÃO HOUVE|[A-Z]{2,3} -|Confederação|\d+\s*/\s*\d+)")


def clean_motivo(raw: str) -> str:
    cleaned = re.sub(r"^[A-Z]\d*(?:\.\d+)*\.?\s*", "", raw.strip())
    return re.sub(r"\s+", " ", cleaned).strip()


def normalize_reason(text: str) -> str:
    text = normalize_text(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def split_reason_components(reason: str) -> tuple[str, str]:
    parts = re.split(r"\s+-\s+", str(reason), maxsplit=1)
    category = parts[0].strip()
    detail = parts[1].strip() if len(parts) > 1 else ""
    return category, detail


def get_section(text: str, start_marker: str, end_marker: str) -> str:
    m = re.search(
        rf"{re.escape(start_marker)}\n(.+?)(?={re.escape(end_marker)})",
        text,
        re.DOTALL,
    )
    return m.group(1) if m else ""


def parse_card_section(
    section_text: str,
    home_team: str,
    away_team: str,
    card_type: str,
) -> list[dict]:
    cards: list[dict] = []
    current_card: dict | None = None
    motivo_lines: list[str] = []
    in_motivo = False

    def flush_current() -> None:
        if current_card and motivo_lines:
            reason = clean_motivo(" ".join(motivo_lines))
            reason_category, reason_detail = split_reason_components(reason)
            cards.append({
                **current_card,
                "reason": reason,
                "reason_key": normalize_reason(reason),
                "reason_category": reason_category,
                "reason_category_key": normalize_reason(reason_category),
                "reason_detail": reason_detail,
                "reason_detail_key": normalize_reason(reason_detail or reason_category),
            })

    for line in section_text.split("\n"):
        line = line.strip()
        if not line or SKIP_RE.match(line):
            continue

        timestamp = TIMESTAMP_RE.match(line)
        if timestamp:
            flush_current()
            current_card = {
                "card_type": card_type,
                "team": identify_team(line, home_team, away_team),
                "minute": int(timestamp.group("minute")),
                "half": timestamp.group("half"),
            }
            motivo_lines = []
            in_motivo = False

        elif "Motivo:" in line:
            in_motivo = True
            raw = re.sub(r"^.*?Motivo:\s*", "", line)
            motivo_lines = [raw] if raw else []

        elif in_motivo and not SKIP_RE.match(line):
            motivo_lines.append(line)

    flush_current()
    return [card for card in cards if card.get("team") and card.get("reason")]


## **5. Construção dos Datasets**

São geradas duas visões complementares:

- `dataset`: formato largo, com duas linhas por partida, uma para cada `focal_team`;
- `card_reasons`: formato longo, com uma linha por cartão, incluindo `card_type`, `minute`, `half`, `team`, `reason_category` e `reason_detail`.

O notebook 2 deve consumir `card_reasons_long.csv`, evitando qualquer parsing de PDF na etapa de modelagem.

In [16]:
def process_match(row: pd.Series, text: str) -> tuple[list[dict], list[dict]]:
    home_team = row["home_team"]
    away_team = row["away_team"]

    yellow_section = get_section(text, "Cartões Amarelos", "Cartões Vermelhos")
    red_section = get_section(text, "Cartões Vermelhos", "Ocorrências / Observações")

    yellow_cards = parse_card_section(yellow_section, home_team, away_team, "yellow")
    red_cards = parse_card_section(red_section, home_team, away_team, "red")
    all_cards = yellow_cards + red_cards

    arbitro = parse_arbitro(text)
    match_date = parse_match_date(row["match_date"])

    team_records = []
    for focal_team in [home_team, away_team]:
        yellow_reasons = [card["reason"] for card in yellow_cards if card["team"] == focal_team]
        red_reasons = [card["reason"] for card in red_cards if card["team"] == focal_team]

        record: dict = {
            "year": row["year"],
            "match_date": match_date,
            "home_team": home_team,
            "away_team": away_team,
            "focal_team": focal_team,
            "arbitro": arbitro,
        }
        for i, reason in enumerate(yellow_reasons, 1):
            record[f"yellow_reason_{i}"] = reason
        for i, reason in enumerate(red_reasons, 1):
            record[f"red_reason_{i}"] = reason

        team_records.append(record)

    card_records = []
    for card in all_cards:
        card_records.append({
            "year": row["year"],
            "round": row["round"],
            "game_id": row["game_id"],
            "game_number": row["game_number"],
            "match_date": match_date,
            "home_team": home_team,
            "away_team": away_team,
            "arbitro": arbitro,
            **card,
        })

    return team_records, card_records


In [17]:
all_records = []
all_card_records = []

for _, row in index.iterrows():
    text = extract_pdf_text(row["file_path"])
    team_records, card_records = process_match(row, text)
    all_records.extend(team_records)
    all_card_records.extend(card_records)

In [18]:
team_game_base_cols = ["year", "match_date", "home_team", "away_team", "focal_team", "arbitro"]
card_reason_cols = [
    "year",
    "round",
    "game_id",
    "game_number",
    "match_date",
    "home_team",
    "away_team",
    "arbitro",
    "card_type",
    "team",
    "minute",
    "half",
    "reason",
    "reason_key",
    "reason_category",
    "reason_category_key",
    "reason_detail",
    "reason_detail_key",
]

dataset = pd.DataFrame(all_records)
card_reasons = pd.DataFrame(all_card_records)

yellow_cols = sorted(
    [c for c in dataset.columns if c.startswith("yellow_reason_")],
    key=lambda c: int(c.split("_")[-1]),
)
red_cols = sorted(
    [c for c in dataset.columns if c.startswith("red_reason_")],
    key=lambda c: int(c.split("_")[-1]),
)

ordered_cols = team_game_base_cols + yellow_cols + red_cols
dataset = dataset.reindex(columns=ordered_cols)
card_reasons = card_reasons.reindex(columns=card_reason_cols).sort_values(
    ["year", "round", "game_number", "card_type", "half", "minute"]
).reset_index(drop=True)

card_reasons.head()

,year,round,game_id,game_number,match_date,home_team,away_team,arbitro,card_type,team,minute,half,reason,reason_key,reason_category,reason_category_key,reason_detail,reason_detail_key
0,2020,1,815072,1,2020-08-09,Flamengo,Atlético Mineiro,Raphael Claus,yellow,Atlético Mineiro,18,1T,Reclamar / protestar (verbalmente ou por gesto...,reclamar / protestar (verbalmente ou por gesto...,Reclamar / protestar (verbalmente ou por gesto...,reclamar / protestar (verbalmente ou por gesto...,Por reclamar ostensivamente com palavras e ges...,por reclamar ostensivamente com palavras e ges...
1,2020,1,815072,1,2020-08-09,Flamengo,Atlético Mineiro,Raphael Claus,yellow,Atlético Mineiro,37,1T,Cometer uma falta tática para impedir um ataqu...,cometer uma falta tatica para impedir um ataqu...,Cometer uma falta tática para impedir um ataqu...,cometer uma falta tatica para impedir um ataqu...,Calçar o adversário disputando a bola e impedi...,calcar o adversario disputando a bola e impedi...
2,2020,1,815072,1,2020-08-09,Flamengo,Atlético Mineiro,Raphael Claus,yellow,Flamengo,18,2T,Dar ou tentar dar uma rasteira ou um calço em ...,dar ou tentar dar uma rasteira ou um calco em ...,Dar ou tentar dar uma rasteira ou um calço em ...,dar ou tentar dar uma rasteira ou um calco em ...,Dar um calço em seu adversário na disputa pela...,dar um calco em seu adversario na disputa pela...
3,2020,1,815072,1,2020-08-09,Flamengo,Atlético Mineiro,Raphael Claus,yellow,Atlético Mineiro,28,2T,"Outro motivo (somente neste caso, abriria um c...","outro motivo (somente neste caso, abriria um c...","Outro motivo (somente neste caso, abriria um c...","outro motivo (somente neste caso, abriria um c...",Por cometer uma atitude antidesportiva fora da...,por cometer uma atitude antidesportiva fora da...
4,2020,1,815072,1,2020-08-09,Flamengo,Atlético Mineiro,Raphael Claus,yellow,Flamengo,28,2T,Dar ou tentar dar uma rasteira ou um calço em ...,dar ou tentar dar uma rasteira ou um calco em ...,Dar ou tentar dar uma rasteira ou um calço em ...,dar ou tentar dar uma rasteira ou um calco em ...,Dar um calço em seu adversário na disputa pela...,dar um calco em seu adversario na disputa pela...


## **6. Validação**

Verificação rápida de cobertura: quantos registros têm árbitro preenchido, quantos têm pelo menos um cartão amarelo/vermelho, distribuição de cartões por ano e integridade da tabela longa.

In [19]:
validation_summary = pd.Series({
    "team_game_rows": len(dataset),
    "card_reason_rows": len(card_reasons),
    "referee_filled_rows": dataset["arbitro"].notna().sum(),
    "team_rows_with_yellow": dataset[yellow_cols].notna().any(axis=1).sum(),
    "team_rows_with_red": dataset[red_cols].notna().any(axis=1).sum(),
    "unique_reason_keys": card_reasons["reason_key"].nunique(),
})
validation_summary

team_game_rows            4854
card_reason_rows         10963
referee_filled_rows       4854
team_rows_with_yellow     4382
team_rows_with_red         458
unique_reason_keys        4922
dtype: int64

In [20]:
yellow_count = dataset[yellow_cols].notna().sum(axis=1)
red_count = dataset[red_cols].notna().sum(axis=1)

stats = (
    dataset.assign(n_yellow=yellow_count, n_red=red_count)
    .groupby("year")[["n_yellow", "n_red"]]
    .mean()
    .round(2)
)
stats.columns = ["média amarelos/time", "média vermelhos/time"]
stats

,média amarelos/time,média vermelhos/time
year,,
2020,1.98,0.11
2021,2.06,0.09
2022,2.24,0.12
2023,2.28,0.10
2024,2.27,0.11
2025,2.15,0.10
2026,2.02,0.12


## **7. Salvamento**

Arquivos gerados:

- `data/processed/sumulas_clean.csv`: base larga por time-jogo;
- `data/processed/card_reasons_long.csv`: base longa por cartão, usada no notebook 2.

In [21]:
dataset.to_csv(OUTPUT_PATH, index=False)
card_reasons.to_csv(CARD_REASONS_LONG_PATH, index=False)
{
    "sumulas_clean": dataset.shape,
    "card_reasons_long": card_reasons.shape,
}

{'sumulas_clean': (4854, 17), 'card_reasons_long': (10963, 18)}